# Chronological Train/Test Split

## Objective

This notebook creates a leakage-safe train/test split for
Top-N recommendation evaluation.

We will:
- Perform per-user chronological splitting
- Ensure no future information leaks into training
- Validate split integrity
- Export train/test datasets

All model statistics must be computed on TRAIN only.

## Imports and Configuration
Define split parameters.

In [1]:
import pandas as pd

TEST_RATIO = 0.2
MIN_TEST_INTERACTIONS = 1  # at least 1 interaction in test
RANDOM_STATE = 42

In [2]:
class Dataset():
    def __init__(self, path):
        self.path = path
        self.dataset = pd.read_csv(self.path)

## Load Processed Dataset

In [3]:
rating_dataset = Dataset("../data/processed/ratings_clean.csv")

In [6]:
rating_dataset.dataset.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,2000-07-30 18:45:03
1,1,3,4.0,2000-07-30 18:20:47
2,1,6,4.0,2000-07-30 18:37:04
3,1,47,5.0,2000-07-30 19:03:35
4,1,50,5.0,2000-07-30 18:48:51


## Splitting Strategy

We perform a per-user chronological split:

For each user:
- Sort interactions by time
- First (1 - TEST_RATIO) → Train
- Last TEST_RATIO → Test

This simulates real-world recommendation:
We train on past behavior and predict future behavior.

In [4]:
train_list = []
test_list = []

for user, user_df in rating_dataset.dataset.groupby("userId"):
    user_df = user_df.sort_values("timestamp")
    
    n_interactions = len(user_df)
    n_test = max(int(n_interactions * TEST_RATIO), MIN_TEST_INTERACTIONS)
    
    test_part = user_df.tail(n_test)
    train_part = user_df.iloc[:-n_test]
    
    # Only include users with at least 1 train interaction
    if len(train_part) > 0:
        train_list.append(train_part)
        test_list.append(test_part)

train_df = pd.concat(train_list)
test_df = pd.concat(test_list)

In [11]:
train_df.head()

,userId,movieId,rating,timestamp
73,1,1210,5.0,2000-07-30 18:08:19
43,1,804,4.0,2000-07-30 18:08:19
120,1,2018,5.0,2000-07-30 18:08:43
171,1,2628,4.0,2000-07-30 18:08:43
183,1,2826,4.0,2000-07-30 18:08:43


In [12]:
test_df.head()

,userId,movieId,rating,timestamp
176,1,2654,5.0,2000-07-30 18:56:33
174,1,2644,4.0,2000-07-30 18:56:33
76,1,1219,2.0,2000-07-30 18:56:33
91,1,1348,4.0,2000-07-30 18:56:33
180,1,2716,5.0,2000-07-30 18:56:54


## Validate Split Integrity

We verify:
- No overlap between train and test
- All test timestamps are after train timestamps
- No user exists only in test

In [5]:
train_pairs = set(zip(train_df.userId, train_df.movieId))
test_pairs = set(zip(test_df.userId, test_df.movieId))

overlap = train_pairs.intersection(test_pairs)
print("Overlap size:", len(overlap))

Overlap size: 0


In [6]:
leakage_count = 0

for user in test_df["userId"].unique():
    max_train_time = train_df[train_df.userId == user]["timestamp"].max()
    min_test_time = test_df[test_df.userId == user]["timestamp"].min()
    
    if min_test_time < max_train_time:
        leakage_count += 1

print("Users with temporal leakage:", leakage_count)

Users with temporal leakage: 0


## Post-Split Statistics

We examine:
- Number of users
- Train/test interaction counts
- Train/test ratio

In [8]:
print("Train interactions:", len(train_df))
print("Test interactions:", len(test_df))

print("Train users:", train_df["userId"].nunique())
print("Test users:", test_df["userId"].nunique())

Train interactions: 80896
Test interactions: 19940
Train users: 610
Test users: 610


In [7]:
print("Avg train interactions per user:", train_df.groupby("userId").size().mean())
print("Avg test interactions per user:", test_df.groupby("userId").size().mean())

Avg train interactions per user: 132.61639344262295
Avg test interactions per user: 32.68852459016394


## Prepare Evaluation Artifacts

We build:
- Ground truth dictionary for test items
- User → train items mapping (for filtering seen items)

In [9]:
ground_truth = (
    test_df.groupby("userId")["movieId"]
    .apply(list)
    .to_dict()
)

user_train_items = (
    train_df.groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

In [ ]:
# Ground truth for userId 2
ground_truth.get(2)

[89774, 1704, 122882, 114060, 80489]

In [ ]:
# Training samples for userId 2
user_train_items.get(2)

{318,
 333,
 3578,
 6874,
 8798,
 46970,
 48516,
 58559,
 60756,
 68157,
 71535,
 74458,
 77455,
 79132,
 80906,
 86345,
 91529,
 91658,
 99114,
 106782,
 109487,
 112552,
 115713,
 131724}

## Save Train/Test Datasets

In [16]:
train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

## Save evaluation artifacts

In [18]:
import pickle

with open("../data/processed/ground_truth.pkl", "wb") as f:
    pickle.dump(ground_truth, f)

with open("../data/processed/user_train_items.pkl", "wb") as f:
    pickle.dump(user_train_items, f)